# YOLO Inference Script

This script provides a simple way to perform inference on images using the YOLO model and save the results in a structured format.

---

## 🚀 **Setup**

### **1. Install dependencies**

Make sure you have Python installed. Then, install the required libraries:

```bash
pip install ultralytics pillow
```

### **2. Prepare the model**

Download the YOLO model weights file you want to use and note its path.

### **3. Organize your data**

Place the input images in a directory (e.g., `input_images`).

---

## ⚙️ **Script Usage**

### **Inputs**

- `input_dir`: Directory containing the input images.
- `output_dir`: Directory where output images and label files will be saved.
- `model_path`: Path to the YOLO model weights file.
- `imgsz`: Image size for inference (default: 640).
- `conf`: Confidence threshold for predictions (default: 0.50).
- `normalize`: Whether to save normalized coordinates (default: `True`).

### **Example**

```python
run_inference(
    input_dir='input_images',
    output_dir='output_results',
    model_path='yolov8.pt',
    imgsz=640,
    conf=0.5,
    normalize=True
)
```

---

## 📂 **Outputs**

### **1. Images**

Annotated images with predictions are saved in `output_dir/images`.

### **2. Labels**

Label files are saved in `output_dir/labels`. Each label file contains lines in the following format:

```text
class_id x1 y1 x2 y2 x3 y3 x4 y4
```

where `x1, y1, ..., x4, y4` are the bounding box coordinates, either normalized or absolute, based on the `normalize` parameter.


---

## 🖥️ **Run from the Terminal**

You can also run the script directly from the terminal using the YOLO Command Line Interface (CLI). Below is a generic command:

```bash
yolo mode=predict model="path/to/model.pt" source="path/to/image_directory" imgsz=640 conf=0.5 save_txt=True
```

Make sure to replace `"path/to/model.pt"` and `"path/to/image_directory"` with the appropriate paths on your system.

For more information on using the YOLO CLI, check out the [CLI usage guide](https://docs.ultralytics.com/usage/cli/).

For more details on inference arguments and how to modify the code, refer to the [official Ultralytics documentation](https://docs.ultralytics.com/modes/predict/#inference-arguments).

---






In [ ]:
import os
from ultralytics import YOLO
from PIL import Image

def is_image_file(filename: str) -> bool:
    """
    Check if the file is a valid image file based on its extension.

    Args:
        filename (str): The name of the file to check.

    Returns:
        bool: True if the file has a valid image extension, False otherwise.
    """
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    return any(filename.lower().endswith(ext) for ext in valid_extensions)

def run_inference(
    input_dir: str, 
    output_dir: str, 
    model_path: str, 
    imgsz: int = 640, 
    conf: float = 0.50, 
    normalize: bool = True
) -> None:
    """
    Run inference on images in the input directory using the YOLO model and save results as .txt files
    with oriented bounding boxes (xyxyxyxy format), optionally normalized, and including the class.

    Args:
        input_dir (str): Path to the input directory containing images.
        output_dir (str): Path to the output directory where images and labels will be saved.
        model_path (str): Path to the YOLO model weights file.
        imgsz (int): Size of the image for inference (default: 640).
        conf (float): Confidence threshold for predictions (default: 0.50).
        normalize (bool): Whether to save normalized coordinates or absolute pixel coordinates (default: True).

    Returns:
        None
    """
    # Load the pretrained YOLO model
    model = YOLO(model_path)

    # Ensure the output directories for images and labels exist
    labels_dir = os.path.join(output_dir, 'labels')
    images_dir = os.path.join(output_dir, 'images')
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(images_dir, exist_ok=True)

    # Get list of image files in the input directory
    image_files = sorted([f for f in os.listdir(input_dir) if is_image_file(f)])

    # Process each image file
    for image_file in image_files:
        image_path = os.path.join(input_dir, image_file)

        # Check if the file is a valid image
        if not os.path.isfile(image_path):
            continue

        try:
            # Run inference on the image with the specified image size and confidence threshold
            results = model.predict(source=image_path, imgsz=imgsz, conf=conf, save=False)  # save=False to avoid automatic saving by YOLO
        except FileNotFoundError:
            print(f"File not found or invalid image: {image_path}")
            continue

        # Process each result for the current image
        for result in results:
            # Save the image with predictions
            output_image_path = os.path.join(images_dir, image_file)
            result_image = result.plot()
            im_rgb = Image.fromarray(result_image[..., ::-1])  # Convert BGR to RGB
            im_rgb.save(output_image_path)
            print(f"Saved predicted image for {image_file} in {images_dir}")

            # Prepare label file path
            base_filename = os.path.splitext(image_file)[0]
            output_label_file_path = os.path.join(labels_dir, f"{base_filename}.txt")

            # Write all OBB detections to the label file
            with open(output_label_file_path, 'w') as f:
                if result.obb is not None and len(result.obb) > 0:
                    # Iterate over all OBB detections
                    for obb in result.obb:
                        # Get the class, confidence, and coordinates
                        class_id = int(obb.cls.item())
                        #coordinates = obb.xyxyn.cpu().numpy().flatten() if normalize else obb.xyxyxyxy.cpu().numpy().flatten()
                        xywhr = obb.xywhr.cpu().numpy().flatten()
                        angle = xywhr[4]
                        coordinates = obb.xyxyxyxyn.cpu().numpy().flatten() if normalize else obb.xyxy.cpu().numpy().flatten()
                        confidence = obb.conf.item()

                        # Write the class and coordinates to the file
                        f.write(f"{class_id} {' '.join(map(str, coordinates))} {angle} {confidence}\n")

            print(f"Processed and saved labels for {image_file} in {labels_dir}")


In [ ]:
# FIRST STEP
image_directory = "/Users/jocareher/Downloads/new_dataset/obbabyface_rot_adults_vs_children/test/images" # Path to your input images directory
output_directory = "/Users/jocareher/Documents/obbabyface_06"  # Path to your output directory for labels

run_inference(image_directory, output_directory, model_path="/Users/jocareher/Library/CloudStorage/OneDrive-Personal/Educacion/PhD_UPF_2023/models_weights/obbabyface_weights.pt", normalize=False)






image 1/1 /Users/jocareher/Downloads/new_dataset/obbabyface_rot_adults_vs_children/test/images/000cc0cd61c6c8d7.jpg: 640x448 (no detections), 111.3ms
Speed: 2.1ms preprocess, 111.3ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 448)
Saved predicted image for 000cc0cd61c6c8d7.jpg in /Users/jocareher/Documents/obbabyface_06/images
Processed and saved labels for 000cc0cd61c6c8d7.jpg in /Users/jocareher/Documents/obbabyface_06/labels

image 1/1 /Users/jocareher/Downloads/new_dataset/obbabyface_rot_adults_vs_children/test/images/00824274fbef9c93.jpg: 640x448 (no detections), 102.9ms
Speed: 1.3ms preprocess, 102.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 448)
Saved predicted image for 00824274fbef9c93.jpg in /Users/jocareher/Documents/obbabyface_06/images
Processed and saved labels for 00824274fbef9c93.jpg in /Users/jocareher/Documents/obbabyface_06/labels

image 1/1 /Users/jocareher/Downloads/new_dataset/obbabyface_rot_adults_vs_children/test/images/00b2